# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.11 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

Lecture 01 - In this notebook under task 1.1 I counted the lines from the terminal which we learned in lecture 1 as well as python basics like printing (which is used throughout this notebook).

Lecutre 02 - In this notebook I used dictionaries to store key-value pairs like the "patient_id" and "visit" pairs, this was done in task 2.3 and 3.2. 

Lecture 03 - In task 4.3, I use NumPy to convert the complete_pairs,to get the mean per visit across patients, NumPy is a package that allows us to calculate multiple things at one time. I used the the axis=0 because it would look at all the rows vertically from the patients and average the values to produce one value for each visit. 

Lecture 04 - I used pandas in various places throughout this notebook like in task 5.1, I made a series where the index labels named each check and then showed if the checks passed or failed. 

Lecture 05 - In this lecture we discussed a lot about handling missing data and in several tasks throughout this notebook I used that, for example in task 3.1 I used it to take the 9999 values and handle them as missing. 

Lecture 06 - In this lecture we talked about joins/merges and in task 4.2 I used the outer merge to merge visits_clean and a1c_long to be able to count the visits with or without the a1c results. 


## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.

def profile(name, table):
    return{
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }

results = []
for name in manifest["files"]:
    results.append(profile(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index=False)
table_profile

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

### Cleaned Tables

#### patients_clean
- Gain: one row to each unique patient
- Key: patient_id
- Violations: duplicated rows for each patient or invalid values


#### sites_clean
- Gain: each row is for each site records
- Key: sites_code
- Violations: duplicated rows or invalid values

#### visits_clean
- Gain: one row per patient visit collected at baseline or at week 12
- Key: visit and patient_id 
- Violations: duplicated rows or rechecks during week 12 

#### a1c
- Gain: one row per patient per visit
- Key: visit and patient_id 
- Violations: duplicated rows


### Cardinality: 

This is the code from 4.4 that goes over the cardinality:
analysis = visits_clean.merge(patients_clean, on = "patient_id", how = "left", validate = "many_to_one")
analysis = analysis.merge(sites_current, on = "site_code", how = "left", validate = "many_to_one")
analysis = analysis.merge(a1c_long, on = ["patient_id", "visit"], how = "left", validate = "many_to_many")


What this cardinality means is how the variables are connected to each other: 
- visits to patient_id is many to one
- visits to sites_code is many to one
- patient_id and visit to a1c is many to many

## Stacking and later recheck 
Each patient's visit has info about patient_id, visit, visit_date, sbp_mmhg, weight_kg, a1c_pct, gender, birth_year, site_code, site_name,source_file. 

### 2.2 Stack the two visit files

In [4]:
south_visits = raw["visits_south"].copy(deep = True).rename(columns={"wt_kg": "weight_kg"})
north_visits = raw["visits_north"].copy(deep = True)
north_visits["source_file"] = "visits_north.csv"
south_visits["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north_visits, south_visits], ignore_index = True)
visits_raw.shape
print(visits_raw)

      patient_id     visit  visit_date visit_time sbp_mmhg weight_kg  \
0         P00002  baseline  2026-02-20      09:50      132      85.2   
1         P00002    week12  2026-05-08      15:20      129      82.0   
2         P00003  baseline  2026-01-16      11:45      148      67.5   
3         P00003      wk12  2026-04-07      10:00      137      64.0   
4         P00004  baseline  2026-02-25      10:55      145     106.2   
...          ...       ...         ...        ...      ...       ...   
57366     P24976    week12  2026-05-14      10:50      153      78.3   
57367     P19276  baseline  2026-01-22      15:35      160     122.2   
57368     P21334    week12  2026-05-05      09:40      129      98.2   
57369     P03524  baseline  2026-02-06      10:05      160     117.6   
57370     P17358  baseline  2026-03-17      08:30      171     109.8   

            source_file  
0      visits_north.csv  
1      visits_north.csv  
2      visits_north.csv  
3      visits_north.csv  
4    

### 2.3 Audit the raw tables

In [5]:
issue_audit = [
    "patients: gender values that are not a code",
    "patients: empty gender",
    "patients: birth_year sentinel 9999",
    "patients: site codes needing normalization",
    "sites: rows with a repeated site_code",
    "visits: visit labels that are not baseline or week12",
    "visits: sbp sentinel 999",
    "visits: sbp outside 70 through 250",
    "visits: date parse failures",
    "visits: rows in repeated patient_id and visit pairs"
]

visit_codes = {
    "baseline": "baseline",
    "bl" : "baseline",
    "week 12" : "week12",
    "week12": "week12",
    "wk12": "week12",
}

count = [
    ((raw["patients"]["gender"].str.strip() != "") & ~raw["patients"]["gender"].isin(["F","M", "X"])).sum(),
    (raw["patients"]["gender"].str.strip() == "").sum(),
    (raw["patients"]["birth_year"] == "9999").sum(),
    (raw["patients"]["site_code"] != raw["patients"]["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")).sum(),
    raw["sites"].duplicated(subset=["site_code"], keep = False).sum(),
    (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
    (visits_raw["sbp_mmhg"] == "999").sum(),
    ((pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") != 999)
     & ~pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce").between(70, 250)
    & pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce").notna()).sum(),
    (~(visits_raw["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
       & pd.to_datetime(visits_raw["visit_date"], format="%Y-%m-%d", errors="coerce").notna())).sum(),
    pd.DataFrame({"patient_id":visits_raw["patient_id"], "visit" : visits_raw["visit"].str.strip().str.lower().map(visit_codes)}).duplicated(keep = False).sum()
]
issue_audit = pd.DataFrame({"issue": issue_audit, "count": count})
issue_audit.to_csv(AUDIT_PATH, index=False) 
issue_audit

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

## Task 3: Clean

### 3.1 Clean the patients

In [6]:
display(raw["patients"]["gender"].str.strip().str.lower().value_counts(dropna = False))

gender_codes = {
    "female": "F", "f": "F", "woman" : "F",
    "male": "M", "m": "M", "man" : "M",
    "nonbinary": "X", "x": "X", "non-binary" : "X",
}

# remove exact duplicate rows
patients_clean = raw["patients"].copy(deep=True).drop_duplicates()

# gender - strip, lowercase, map to its code, empty values
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map(gender_codes).astype("category")

# birth year (dtype:Int64)
birth = pd.to_numeric(patients_clean["birth_year"], errors="coerce")
patients_clean["birth_year"] = birth.where(birth != 9999).astype("Int64")

# renumber rows
patients_clean = patients_clean.reset_index(drop=True)

gender
f             12075
m             11732
male           1917
female         1827
woman           531
                486
x               434
unknown         378
man             329
nonbinary       118
non-binary      114
nb              104
Name: count, dtype: int64[pyarrow]

In [21]:
display(patients_clean["gender"].cat.categories)
patients_summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean),
            patients_clean["birth_year"].isna().sum(),
            patients_clean["birth_year"].min(),
            patients_clean["birth_year"].max(),
            patients_clean["site_code"].nunique(),
            (patients_clean["site_code"] == "N01").sum(),
            (patients_clean["site_code"] == "S02").sum(),
            (patients_clean["site_code"] == "E04").sum()
            ]
})
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

Index(['F', 'M', 'X'], dtype='str')

### 3.2 Clean the visits

In [8]:
visits_clean = visits_raw.copy(deep=True)

visits_clean = visits_clean.drop_duplicates()

display(visits_clean["visit"].str.strip().str.lower().value_counts())
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_codes)

sbp = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = sbp.where(sbp.between(70, 250)).astype("Int64")

visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors = "coerce")

dates = pd.to_datetime(visits_clean["visit_date"], format="%Y-%m-%d", errors="coerce")
exact_text = visits_clean["visit_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
visits_clean["visit_date"] = dates.where(exact_text)


visit
baseline    29221
week12      26882
bl            455
wk12          381
week 12       362
Name: count, dtype: int64[pyarrow]

In [9]:
repeated = visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)
display(visits_clean[repeated].sort_values(by=["patient_id", "visit", "visit_date"]))

visits_clean = visits_clean.sort_values("visit_time", kind = "stable")
visits_clean = visits_clean.drop_duplicates(subset=["patient_id", "visit"], keep="last").reset_index(drop=True)


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
...,...,...,...,...,...,...,...
55473,P28232,baseline,2026-01-12,10:10,159,61.0,visits_south.csv
56248,P28953,baseline,2026-01-21,10:45,151,74.8,visits_south.csv
56249,P28953,baseline,2026-01-21,10:30,153,74.8,visits_south.csv
25464,P29170,baseline,2026-03-02,11:20,130,78.0,visits_north.csv


In [10]:
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]

visits_summary = pd.DataFrame({
    "metric": ["visits", "baseline visits", "week12 visits", "visit_date missing", "visit_date min", 
               "visit_date max", "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max", "weight_kg missing",
               "weight_kg mean", "visits from visits_north.csv", "visits from visits_south.csv"],

    "values": [len(visits_clean),
               (visits_clean["visit"] == "baseline").sum(),
               (visits_clean["visit"] == "week12").sum(),
                visits_clean["visit_date"].isna().sum(),
                visits_clean["visit_date"].min(),
                visits_clean["visit_date"].max(),
                visits_clean["sbp_mmhg"].isna().sum(),
                visits_clean["sbp_mmhg"].min(),
                visits_clean["sbp_mmhg"].max(),
                visits_clean["weight_kg"].isna().sum(),
                (round(visits_clean["weight_kg"].mean(), 2)),
                (visits_clean["source_file"] == "visits_north.csv").sum(),
                (visits_clean["source_file"] == "visits_south.csv").sum()],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary


,metric,values
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05 00:00:00
5,visit_date max,2026-07-06 00:00:00
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [11]:
decision_table = pd.DataFrame([
    ["patients", "exact duplicate rows", "this is where I dropped the duplicate rows and only kept the first row", "Keeping the duplicated rows would have altered the data and the results as their would be multiple counts of the same patient"],
    ["patients","gender spellings that are not a code", "here I fixed the gender spellings to be consistent to F, M, X", "this was done because we need to have consistent values for the same variable"],
    ["patients", "site codes needing normalization", "here I fixed the site codes by removing the spaces and -", "similar to the gender there needs to be consistency for it to be easy to understand the results."],
    ["patients", "birth_year sentinel 9999", "I fixed the values that had 9999 as a place holder for the missing birth year to be NA", "this was done because the year 9999 isn't there and it needed to be counted among the missing to be able to analyze accurately."],
    ["visits", "weight column named wt_kg in visits_south.csv", "the column for weight was renamed to weight_kg", "this was done to include the units of the weight to be clear it is in kg and not in pounds."],
    ["visits", "sbp sentinel 999 and readings outside 70 through 250", "The values were fixed to be missing if they were outside the given range of 70-250", "This was done to have the accurate values for sbp and not include extreme outliers"],
    ["visits", "impossible and loosely written visit dates", "this was to have a consistent date throughout all the values and not include those that didn't follow this", "This was done to again have consistency throughout the date values"],
    ["visits", "repeated patient_id and visit pairs", "this was to keep the most recentreadings from the patients", "I did this to only have one reading for each patient and visit pair to not duplicate or count the same reading twice"],
    ["sites", "repeated site_code rows", "Here I only kept the first row", "This was so there wouldn't be multiple rows of the same site code."],
],
    columns=["table", "issue", "decision", "reason"])

decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table



,table,issue,decision,reason
0,patients,exact duplicate rows,this is where I dropped the duplicate rows and...,Keeping the duplicated rows would have altered...
1,patients,gender spellings that are not a code,here I fixed the gender spellings to be consis...,this was done because we need to have consiste...
2,patients,site codes needing normalization,here I fixed the site codes by removing the sp...,similar to the gender there needs to be consis...
3,patients,birth_year sentinel 9999,I fixed the values that had 9999 as a place ho...,this was done because the year 9999 isn't ther...
4,visits,weight column named wt_kg in visits_south.csv,the column for weight was renamed to weight_kg,this was done to include the units of the weig...
5,visits,sbp sentinel 999 and readings outside 70 throu...,The values were fixed to be missing if they we...,This was done to have the accurate values for ...
6,visits,impossible and loosely written visit dates,this was to have a consistent date throughout ...,This was done to again have consistency throug...
7,visits,repeated patient_id and visit pairs,this was to keep the most recentreadings from ...,I did this to only have one reading for each p...
8,sites,repeated site_code rows,Here I only kept the first row,This was so there wouldn't be multiple rows of...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:
# part 1
a1c_long = raw["a1c_wide"].melt(
    id_vars = ["patient_id"],
    value_vars = ["a1c_baseline", "a1c_week12"],
    var_name = "visit",
    value_name = "a1c_pct",
)

# part 2
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})

# part 3
a1c_long = a1c_long.dropna(subset=["a1c_pct"])

a1c_summary = pd.DataFrame({
    "visit":["baseline", "week12"],
    "results": [(a1c_long["visit"] == "baseline").sum(),
                (a1c_long["visit"] == "week12").sum()],
    "mean_a1c_pct": [round(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean(), 2),
                     round(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean(), 2)],
})
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)

### 4.2 Check the keys before joining

In [13]:
sites = raw["sites"]

try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError:
    repeated_sites = sites[sites.duplicated(subset=["site_code"], keep=False)]
    display(repeated_sites.sort_values(by="site_code"))
sites_current = sites[sites["record_status"] == "current"][["site_code", "site_name"]]
sites_current

,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
3,S02,South Valley Clinic
4,W03,West Clinic


In [14]:
check = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer", indicator=True)   
display(check["_merge"].value_counts())
merge_audit = pd.DataFrame({
    "check":[
       "visits with an A1c result",
       "visits without an A1c result",
       "A1c results without a visit",
       "visits whose patient is not in the registry",
       "registry patients with no visit",
       "registry patients whose site is not in the site list",
       "site list rows removed by the current-record rule",
    ],
    "count": [
        (check["_merge"] == "both").sum(),
        (check["_merge"] == "left_only").sum(),
        (check["_merge"] == "right_only").sum(),
        (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
        (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
        (~patients_clean["site_code"].isin(sites["site_code"])).sum(),
        len(sites) - len(sites_current),
    ],
})
merge_audit["count"] = merge_audit["count"].astype(int)
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,2334
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [15]:
sbp_wide = visits_clean.pivot(index = "patient_id", columns = "visit", values = "sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index = False)
sbp_wide_sample

,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [16]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()

arr = np.array(complete_pairs, dtype=float)
display(arr.shape)

visit_means = arr.mean(axis=0)
visit_means

(27149, 2)

array([143.01337066, 137.15061328])

### 4.4 Join the analysis table

In [17]:
analysis = visits_clean.merge(patients_clean, on = "patient_id", how = "left", validate = "many_to_one")
analysis = analysis.merge(sites_current, on = "site_code", how = "left", validate = "many_to_one")
analysis = analysis.merge(a1c_long, on = ["patient_id", "visit"], how = "left", validate = "many_to_many")

analysis = analysis[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct", "gender", "birth_year", "site_code", "site_name", "source_file"]]

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})

print(len(analysis), len(visits_clean))

analysis.shape


57181 57181


(57181, 11)

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [18]:
checks = pd.Series({
    "analysis has as many rows as visits_clean":len(analysis) == len(visits_clean),
    "no patient id and visit pair repeats in analysis":not analysis.duplicated(subset = ["patient_id", "visit"]).any(),
    "every sbp mmhg that is present in analysis is from 70 through 250":analysis["sbp_mmhg"].dropna().between(70, 250).all(),
    "every site name that is present is a current site name":analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
    "sbp_wide patient_id is unique":sbp_wide["patient_id"].is_unique,
    "every raw table still equals its snapshot":all(raw[name].equals(raw_snapshot[name]) for name in raw)
}).astype(bool)

display(checks)
assert checks.all(), checks[~checks]

analysis has as many rows as visits_clean                            True
no patient id and visit pair repeats in analysis                     True
every sbp mmhg that is present in analysis is from 70 through 250    True
every site name that is present is a current site name               True
sbp_wide patient_id is unique                                        True
every raw table still equals its snapshot                            True
dtype: bool

### 5.2 Save the analysis extract

In [19]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop = True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)

print(len(analysis_sample), visits_clean["patient_id"].isin(SAMPLE_IDS).sum())
analysis_sample

28 28


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N-01,<NA>,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N-01,<NA>,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [20]:
round_trip = pd.read_csv(ANALYSIS_SAMPLE_PATH,
    dtype = {
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
        
    },
    parse_dates=["visit_date"]
)
print(round_trip.equals(analysis_sample))

True
